# Session 03 · Correlation and Regression

Run each cell with **Shift+Enter**. Original material by Aseef Ahmed.

## Data: students (synthetic)

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
rng = np.random.default_rng(3)
n = 200
df = pd.DataFrame({"attendance": rng.uniform(50, 100, n).round(0), "study_hours": rng.uniform(0, 15, n).round(1),
                   "campus": rng.choice(["North", "South"], n)})
df["marks"] = (5 + 0.5 * df["attendance"] + 1.8 * df["study_hours"] + (df["campus"] == "North") * 3
               + rng.normal(0, 6, n)).round(1)
x, y = df["attendance"].to_numpy(), df["marks"].to_numpy()

## Example 1 - covariance and Pearson r by hand

In [2]:
cov = ((x - x.mean()) * (y - y.mean())).sum() / (len(x) - 1)
r = cov / (x.std(ddof=1) * y.std(ddof=1))
r_scipy, p = stats.pearsonr(x, y)
print(f"cov {cov:.2f} | r {r:.3f} | scipy r {r_scipy:.3f}, p = {p:.2e}")
assert np.isclose(r, r_scipy)

cov 77.36 | r 0.499 | scipy r 0.499, p = 5.77e-14


## Example 2 - Pearson vs Spearman with an outlier

In [3]:
xs, ys = x[:30], y[:30]
x2, y2 = np.append(xs, 40), np.append(ys, 150)
print("Pearson:", round(stats.pearsonr(xs, ys)[0], 3), "->", round(stats.pearsonr(x2, y2)[0], 3))
print("Spearman:", round(stats.spearmanr(xs, ys)[0], 3), "->", round(stats.spearmanr(x2, y2)[0], 3))

Pearson: 0.466 -> -0.146
Spearman: 0.469 -> 0.331


## Example 3 - zero correlation, strong relationship

In [4]:
u = np.linspace(-3, 3, 100); v = u ** 2
print("r(u, u^2) =", round(stats.pearsonr(u, v)[0], 6))

r(u, u^2) = 0.0


## Example 4 - simple regression with linregress

In [5]:
res = stats.linregress(df["attendance"], df["marks"])
print(f"marks = {res.intercept:.2f} + {res.slope:.3f} * attendance | R2 = {res.rvalue ** 2:.3f} | p = {res.pvalue:.2e}")

marks = 26.60 + 0.427 * attendance | R2 = 0.249 | p = 5.77e-14


## Example 5 - statsmodels table and intervals

In [6]:
model = smf.ols("marks ~ attendance", data=df).fit()
print(model.summary().tables[1])
print(model.conf_int().round(3))
print(model.get_prediction(pd.DataFrame({"attendance": [90]})).summary_frame(alpha=0.05).round(2))

                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     26.5989      4.038      6.587      0.000      18.635      34.563
attendance     0.4272      0.053      8.093      0.000       0.323       0.531
                 0       1
Intercept   18.635  34.563
attendance   0.323   0.531
    mean  mean_se  mean_ci_lower  mean_ci_upper  obs_ci_lower  obs_ci_upper
0  65.05     1.05          62.98          67.12         45.18         84.91


## Example 6 - multiple regression and residuals

In [7]:
m2 = smf.ols("marks ~ attendance + study_hours + C(campus)", data=df).fit()
print(m2.params.round(3)); print("adj R2:", round(m2.rsquared_adj, 3))
print("Shapiro p-value of residuals:", round(stats.shapiro(m2.resid).pvalue, 3))
assert m2.rsquared_adj > model.rsquared_adj

Intercept             8.938
C(campus)[T.South]   -3.617
attendance            0.496
study_hours           1.862
dtype: float64
adj R2: 0.74
Shapiro p-value of residuals: 0.614
